[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C52_Industrial_Research_Practice_Course/03_export_runtime/03_export_runtime.ipynb)

# 03 · 模型导出与推理运行时（迷你 IR / 形状推断 / 动态轴 / 算子覆盖 / 量化对拍）

目标：手写一个**迷你 ONNX**——图 IR、opset 版本、形状推断、动态轴、算子覆盖检查——
并复现导出的四类经典失败：**形状写死、控制流固化、算子不支持、静默回退**。

路线：迷你 IR 与图 → 形状推断 → **不设动态轴的后果**（换 batch 就崩）→
控制流固化 → opset 兼容性检查 → 算子覆盖与静默回退 →
多形状对拍与边界测试 → 量化对拍的正确容差 → ✏️ 练习 → 📖 答案 → 🧪 导出前检查器。

> 心智模型：**导出 = 把一段 Python 程序编译成受限 IR 上的静态图。
> 最有效的通用招数只有一条：把动态部分挪到模型之外。**

## 1 · 迷你 IR：节点、边、opset、符号形状

In [ ]:
import numpy as np, math, collections, itertools, json
rng = np.random.default_rng(0)

# 算子库：每个 op 记录 (自哪个 opset 起可用, 形状推断函数, 参考实现)
def _shape_matmul(shapes, attrs):
    a, b = shapes
    assert a[-1] == b[-2] or isinstance(a[-1], str) or isinstance(b[-2], str), \
        f'matmul 内维不匹配: {a} @ {b}'
    return tuple(list(a[:-1]) + [b[-1]])

def _shape_same(shapes, attrs):  return shapes[0]
def _shape_reduce_last(shapes, attrs): return tuple(shapes[0][:-1])

OPS = {
    'MatMul':  {'since': 1,  'shape': _shape_matmul,
                'run': lambda ins, at: ins[0] @ ins[1]},
    'Add':     {'since': 1,  'shape': _shape_same,
                'run': lambda ins, at: ins[0] + ins[1]},
    'Relu':    {'since': 1,  'shape': _shape_same,
                'run': lambda ins, at: np.maximum(ins[0], 0)},
    'Gelu':    {'since': 20, 'shape': _shape_same,          # ← 直到 opset 20 才是标准算子
                'run': lambda ins, at: 0.5*ins[0]*(1+np.tanh(
                    math.sqrt(2/math.pi)*(ins[0]+0.044715*ins[0]**3)))},
    'LayerNorm': {'since': 17, 'shape': _shape_same,        # ← opset 17 起
                  'run': lambda ins, at: (lambda x, g, b, e: (x - x.mean(-1, keepdims=True))
                         / np.sqrt(x.var(-1, keepdims=True) + e) * g + b)(
                             ins[0], ins[1], ins[2], at.get('epsilon', 1e-5))},
    'ReduceSum': {'since': 1, 'shape': _shape_reduce_last,
                  'run': lambda ins, at: ins[0].sum(-1)},
    'If':      {'since': 1,  'shape': lambda s, a: s[1],
                'run': None},                                # 控制流：需要子图
    'NonZero': {'since': 9,  'shape': lambda s, a: (len(s[0]), 'nnz'),   # ← 形状依赖**数值**
                'run': None},
}

class Node:
    def __init__(self, op_type, inputs, output, **attrs):
        self.op_type, self.inputs, self.output, self.attrs = op_type, inputs, output, attrs
    def __repr__(self):
        a = f' {self.attrs}' if self.attrs else ''
        return f'{self.output} = {self.op_type}({", ".join(self.inputs)}){a}'

class OnnxModel:
    def __init__(self, opset=17):
        self.opset = opset
        self.nodes, self.inputs, self.outputs, self.initializers = [], {}, [], {}
    def add_input(self, name, shape, dtype='float32'):
        self.inputs[name] = {'shape': tuple(shape), 'dtype': dtype}; return name
    def add_init(self, name, arr):
        self.initializers[name] = np.asarray(arr, dtype=np.float64); return name
    def add_node(self, op_type, inputs, output, **attrs):
        self.nodes.append(Node(op_type, inputs, output, **attrs)); return output
    def op_types(self):
        return sorted({n.op_type for n in self.nodes})
    def __repr__(self):
        head = (f'opset={self.opset}\ninputs: ' +
                ', '.join(f'{k}{v["shape"]}' for k, v in self.inputs.items()))
        return head + '\nnodes:\n' + '\n'.join('  ' + repr(n) for n in self.nodes)

# 造一个小模型：LN(GELU(x@W1)@W2)
m = OnnxModel(opset=17)
m.add_input('x', ('batch', 'seq', 8))
m.add_init('W1', rng.normal(size=(8, 16)) * 0.3)
m.add_init('W2', rng.normal(size=(16, 8)) * 0.3)
m.add_init('g', np.ones(8)); m.add_init('b', np.zeros(8))
m.add_node('MatMul', ['x', 'W1'], 'h1')
m.add_node('Gelu', ['h1'], 'h2')
m.add_node('MatMul', ['h2', 'W2'], 'h3')
m.add_node('LayerNorm', ['h3', 'g', 'b'], 'y', epsilon=1e-5)
m.outputs = ['y']
print(m)
assert m.op_types() == ['Gelu', 'LayerNorm', 'MatMul']
print('\n✅ 迷你 IR 就位：静态拓扑 + opset 版本 + 符号形状（"batch"/"seq"）')

## 2 · 形状推断与动态轴：不设的后果

In [ ]:
def infer_shapes(model, concrete=None):
    '''从输入形状推出每个中间张量的形状。concrete: {符号名: 具体值}（None 表示保持符号）。'''
    concrete = concrete or {}
    def resolve(shape):
        return tuple(concrete.get(d, d) if isinstance(d, str) else d for d in shape)
    env = {k: resolve(v['shape']) for k, v in model.inputs.items()}
    env.update({k: v.shape for k, v in model.initializers.items()})
    for n in model.nodes:
        spec = OPS[n.op_type]
        env[n.output] = spec['shape']([env[i] for i in n.inputs], n.attrs)
    return env

env_sym = infer_shapes(m)
print('符号形状推断:')
for k in ['x', 'h1', 'h2', 'h3', 'y']:
    print(f'  {k:<4s} {env_sym[k]}')
assert env_sym['y'] == ('batch', 'seq', 8)
env_c = infer_shapes(m, {'batch': 4, 'seq': 128})
assert env_c['y'] == (4, 128, 8)
print(f'\n代入 batch=4, seq=128: y{env_c["y"]}  ✅ 动态轴让同一张图接受任意形状')

In [ ]:
# ❌ 不设动态轴：导出时的形状被**写死**
m_static = OnnxModel(opset=17)
m_static.add_input('x', (1, 128, 8))               # ← 追踪时 batch=1, seq=128
for k, v in m.initializers.items(): m_static.add_init(k, v)
for n in m.nodes: m_static.add_node(n.op_type, n.inputs, n.output, **n.attrs)
m_static.outputs = ['y']

def check_input_shape(model, actual):
    '''运行时的形状检查。'''
    declared = model.inputs['x']['shape']
    errs = []
    for i, (d, a) in enumerate(zip(declared, actual)):
        if isinstance(d, str):
            continue                                # 动态轴：任意值都行
        if d != a:
            errs.append(f'dim{i}: 图里写死为 {d}，实际传入 {a}')
    return (not errs), errs

print(f"{'实际输入形状':<18s} {'静态图':<28s} {'动态轴图'}")
for shape in [(1, 128, 8), (8, 128, 8), (1, 64, 8), (3, 333, 8)]:
    ok_s, err_s = check_input_shape(m_static, shape)
    ok_d, _ = check_input_shape(m, shape)
    print(f'{str(shape):<18s} {("✅" if ok_s else "❌ " + err_s[0]):<28s} {"✅" if ok_d else "❌"}')

assert check_input_shape(m_static, (1, 128, 8))[0]
assert not check_input_shape(m_static, (8, 128, 8))[0], '静态图换 batch 就崩'
assert all(check_input_shape(m, s)[0] for s in [(1,128,8),(8,128,8),(3,333,8)])
print('\n⚠️  「用 batch=1, seq=128 的样例导出，线上来 batch=8」是最常见的导出事故。')
print('✅ 凡是会变的维度（batch / 序列长 / 图像尺寸）都必须声明为动态轴。')
print('   真实对应: torch.onnx.export(..., dynamic_axes={"x": {0: "batch", 1: "seq"}})')

## 3 · 控制流固化：导出里这是**正确性**问题

模块 01 里 Python 控制流被固化是性能问题（可以重追踪）；
**在导出里它是正确性问题**——图被交出去后，再也不会重新追踪。

In [ ]:
def python_branch_model(use_relu):
    '''Python bool 决定分支 -> 追踪时定死。'''
    g = OnnxModel(opset=17)
    g.add_input('x', ('batch', 8))
    g.add_init('W', rng.normal(size=(8, 8)) * 0.3)
    g.add_node('MatMul', ['x', 'W'], 'h')
    if use_relu:                                   # ← Python 层面的分支
        g.add_node('Relu', ['h'], 'y')
    else:
        g.add_node('Add', ['h', 'h'], 'y')
    g.outputs = ['y']
    return g

g_true = python_branch_model(True)
g_false = python_branch_model(False)
print('use_relu=True  导出的图:', g_true.op_types())
print('use_relu=False 导出的图:', g_false.op_types())
assert 'Relu' in g_true.op_types() and 'Relu' not in g_false.op_types()
print('⚠️  两张**不同的图**。导出后 use_relu 再也改不了 ——')
print('    如果它本该依赖运行时输入，你导出的就是一个**永远走同一条分支**的错模型。')

# ✅ 正确做法：用 If 算子把两个分支都放进图
def graph_branch_model():
    g = OnnxModel(opset=17)
    g.add_input('x', ('batch', 8))
    g.add_input('cond', ())
    g.add_init('W', rng.normal(size=(8, 8)) * 0.3)
    g.add_node('MatMul', ['x', 'W'], 'h')
    g.add_node('Relu', ['h'], 'br_true')
    g.add_node('Add', ['h', 'h'], 'br_false')
    g.add_node('If', ['cond', 'br_true', 'br_false'], 'y')
    g.outputs = ['y']
    return g

g_if = graph_branch_model()
print('\n图内控制流:', g_if.op_types())
assert 'If' in g_if.op_types() and 'Relu' in g_if.op_types()
print('✅ 一张图涵盖两个分支，运行时按 cond 选择。')
print('   代价：两个分支都要能通过形状推断，且图更复杂、部分优化受限。')

## 4 · opset 兼容性与算子覆盖：部署前的两分钟检查

In [ ]:
def check_opset(model):
    '''每个算子都要求 model.opset >= 它的 since version。'''
    bad = [(n.op_type, OPS[n.op_type]['since'])
           for n in model.nodes if OPS[n.op_type]['since'] > model.opset]
    return (not bad), bad

for opset in [20, 17, 15, 9]:
    mm = OnnxModel(opset=opset)
    mm.nodes = m.nodes; mm.inputs = m.inputs; mm.initializers = m.initializers
    ok, bad = check_opset(mm)
    detail = '' if ok else '  需要: ' + ', '.join(f'{o}>=opset{s}' for o, s in bad)
    print(f'opset={opset:<3d} {"✅ 全部算子可用" if ok else "❌ " + str([b[0] for b in bad])}{detail}')

mm17 = OnnxModel(17); mm17.nodes = m.nodes
assert not check_opset(mm17)[0], 'Gelu 需要 opset>=20'
mm20 = OnnxModel(20); mm20.nodes = m.nodes
assert check_opset(mm20)[0]
print('\n⚠️  opset **不是越新越好**：目标运行时可能只支持到 15。')
print('✅ 正确顺序：先查目标运行时支持的 opset 上限，再据此导出。')

In [ ]:
# 运行时的算子支持表（真实世界里各不相同）
RUNTIME_SUPPORT = {
    'onnxruntime-cpu':  {'MatMul','Add','Relu','Gelu','LayerNorm','ReduceSum','If','NonZero'},
    'tensorrt':         {'MatMul','Add','Relu','LayerNorm'},          # Gelu 需插件；无 NonZero
    'tflite':           {'MatMul','Add','Relu'},                      # 算子集最受限
    'coreml-ane':       {'MatMul','Add','Relu','LayerNorm'},          # ANE 支持的子集
}

def coverage_report(model, runtime):
    sup = RUNTIME_SUPPORT[runtime]
    used = model.op_types()
    unsupported = [o for o in used if o not in sup]
    # 静默回退：不支持的算子会把图切成多个子图，交给别的后端
    n_partitions = 1
    prev_ok = None
    for n in model.nodes:
        ok = n.op_type in sup
        if prev_ok is not None and ok != prev_ok:
            n_partitions += 1
        prev_ok = ok
    return {'runtime': runtime, 'used': used, 'unsupported': unsupported,
            'partitions': n_partitions, 'fully_supported': not unsupported}

print(f"{'运行时':<18s} {'不支持的算子':<26s} {'子图数':>7s} {'结论'}")
for rt in RUNTIME_SUPPORT:
    r = coverage_report(m, rt)
    verdict = '✅ 全图加速' if r['fully_supported'] else \
              (f'⚠️ 切成 {r["partitions"]} 个子图 -> **静默回退**')
    print(f'{rt:<18s} {str(r["unsupported"]):<26s} {r["partitions"]:>7d} {verdict}')

r_trt = coverage_report(m, 'tensorrt')
assert 'Gelu' in r_trt['unsupported'] and r_trt['partitions'] > 1
assert coverage_report(m, 'onnxruntime-cpu')['fully_supported']
print('\n⚠️  **静默回退**：TensorRT 遇到不支持的算子不会报错，而是把图切开、')
print('    不支持的部分交给别的后端（甚至 CPU）。模型能跑、数值也对，')
print('    但**性能可能比不优化还差**（多了大量设备间搬运）。')
print('    症状是「转了 TensorRT 但没变快」，而日志里那行「N subgraphs」才是原因。')
print('✅ 所以「导出后的性能验证」必须与「数值验证」同等对待，且要看**分区日志**。')

### 算子不支持的四条出路（按成本递增）

In [ ]:
def decompose_gelu(model):
    '''出路 ③：把 Gelu 分解成基础算子（这里用 tanh 近似的算子组合示意）。'''
    new = OnnxModel(opset=model.opset)
    new.inputs = dict(model.inputs); new.initializers = dict(model.initializers)
    for n in model.nodes:
        if n.op_type == 'Gelu':
            # 真实分解会用 Mul/Add/Tanh/Pow 等基础算子；这里用 Relu 占位表示「换成被支持的算子」
            new.add_node('Relu', n.inputs, n.output)
        else:
            new.add_node(n.op_type, n.inputs, n.output, **n.attrs)
    new.outputs = list(model.outputs)
    return new

ROUTES = [
    ('① 换算子/分解', lambda mdl: decompose_gelu(mdl), '最低', '首选'),
    ('② 提高 opset', None, '低', '若目标运行时也支持'),
    ('③ 分解成基础算子', lambda mdl: decompose_gelu(mdl), '中', '要自己验证数值'),
    ('④ 自定义算子插件', None, '最高', '真的无法避开时'),
]
m_fixed = decompose_gelu(m)
r_before = coverage_report(m, 'tensorrt')
r_after = coverage_report(m_fixed, 'tensorrt')
print(f'TensorRT 覆盖: 处理前 不支持{r_before["unsupported"]}, {r_before["partitions"]} 个子图')
print(f'              处理后 不支持{r_after["unsupported"]}, {r_after["partitions"]} 个子图')
assert r_after['fully_supported'] and r_after['partitions'] == 1
print('\n✅ 出路 ① 成功率最高却常被跳过。多数「不支持」其实是用了 Python 便利写法。')

# 形状依赖**数值**的算子：图 IR 天然困难
m_nz = OnnxModel(opset=17)
m_nz.add_input('x', ('batch', 8))
m_nz.add_node('NonZero', ['x'], 'idx')
m_nz.outputs = ['idx']
env_nz = infer_shapes(m_nz, {'batch': 4})
print(f'\nNonZero 的输出形状: {env_nz["idx"]}  ← 第二维 "nnz" **依赖输入数值**，推断不出来')
assert 'nnz' in env_nz['idx']
print('⚠️  这类算子（NonZero / Unique / masked_select / 动态 topk）在图 IR 里天然困难。')
print('✅ 通用解法：**把动态部分挪到模型之外** ——')
print('   模型输出固定形状的 mask/分数，由调用方在 Python/C++ 侧做筛选。')

## 5 · 多形状对拍与边界测试

**必须在多组不同形状上对拍**，而不只是导出时用的那一组。

In [ ]:
def run_graph(model, feeds, concrete):
    '''执行迷你图。'''
    env = dict(model.initializers)
    for k, v in feeds.items(): env[k] = v
    for n in model.nodes:
        spec = OPS[n.op_type]
        if spec['run'] is None:
            raise NotImplementedError(f'{n.op_type} 需要特殊处理（控制流/数据依赖形状）')
        env[n.output] = spec['run']([env[i] for i in n.inputs], n.attrs)
    return {o: env[o] for o in model.outputs}

def reference(x, W1, W2, g, b, eps=1e-5):
    h = x @ W1
    h = 0.5*h*(1+np.tanh(math.sqrt(2/math.pi)*(h+0.044715*h**3)))
    h = h @ W2
    return (h - h.mean(-1, keepdims=True))/np.sqrt(h.var(-1, keepdims=True)+eps)*g + b

# **含奇数、极小值、非对齐**的形状
TEST_SHAPES = [(1, 128), (2, 17), (8, 512), (3, 333), (1, 1), (5, 7)]
print(f"{'形状':<14s} {'最大绝对误差':>14s} {'通过'}")
all_ok = True
for bsz, seq in TEST_SHAPES:
    x = rng.normal(size=(bsz, seq, 8))
    got = run_graph(m, {'x': x}, {'batch': bsz, 'seq': seq})['y']
    ref = reference(x, m.initializers['W1'], m.initializers['W2'],
                    m.initializers['g'], m.initializers['b'])
    err = float(np.abs(got - ref).max())
    ok = err < 1e-10
    all_ok &= ok
    print(f'{str((bsz,seq)):<14s} {err:>14.2e} {"✅" if ok else "❌"}')
assert all_ok
print('\n✅ 测试形状要**刻意包含**：batch=1、seq=1、奇数、非 8 倍数、极大值。')
print('   很多运行时（尤其 TensorRT）对对齐有隐含要求，只在非对齐形状上暴露。')
print('⚠️  别用 batch=1 做导出的样例输入 —— batch=1 与 batch>1 可能走不同代码路径。')

## 6 · 量化对拍：容差要按量化误差设

In [ ]:
def quantize_int8(w, per_channel=False):
    '''对称量化：scale = max|w| / 127。'''
    if per_channel:
        s = np.abs(w).max(axis=0, keepdims=True) / 127.0
    else:
        s = np.array(np.abs(w).max() / 127.0)
    s = np.where(s == 0, 1e-12, s)
    q = np.clip(np.round(w / s), -127, 127)
    return q, s

def dequantize(q, s): return q * s

W = m.initializers['W1']
for name, pc in [('per-tensor', False), ('per-channel', True)]:
    q, s = quantize_int8(W, per_channel=pc)
    w_hat = dequantize(q, s)
    rel = float(np.abs(w_hat - W).max() / np.abs(W).max())
    print(f'{name:<14s} 权重最大相对误差 {rel:.4%}')
q_pt, s_pt = quantize_int8(W, False); q_pc, s_pc = quantize_int8(W, True)
err_pt = np.abs(dequantize(q_pt, s_pt) - W).max()
err_pc = np.abs(dequantize(q_pc, s_pc) - W).max()
assert err_pc <= err_pt, 'per-channel 量化误差不高于 per-tensor'
print(f'\n✅ per-channel 误差 {err_pc:.5f} <= per-tensor {err_pt:.5f}（每列各有 scale）')

# ⚠️ 用浮点容差去对拍量化结果必然「失败」—— 但那不是 bug
x = rng.normal(size=(4, 32, 8))
ref = reference(x, W, m.initializers['W2'], m.initializers['g'], m.initializers['b'])
W_q = dequantize(*quantize_int8(W, True))
got = reference(x, W_q, m.initializers['W2'], m.initializers['g'], m.initializers['b'])
print(f'\n量化后最大绝对误差: {np.abs(got-ref).max():.4e}')
print(f'  用 rtol=1e-5 判定: {np.allclose(got, ref, rtol=1e-5, atol=1e-6)}   ← 必然 False')
print(f'  用 rtol=2e-2 判定: {np.allclose(got, ref, rtol=2e-2, atol=2e-2)}')
assert not np.allclose(got, ref, rtol=1e-5, atol=1e-6)
print('\n✅ 量化后的正确验证方式：')
print('   ① 对拍**最终任务指标**（准确率/BLEU）而不是逐元素数值')
print('   ② 逐层看「误差是否在预期量级」而不是「是否 allclose」')
print('   ③ 重点找**异常层**（误差比其他层大一个数量级 -> 它的激活有长尾）')

In [ ]:
# 找异常层：逐层量化误差的分布
def per_layer_quant_error(weights, per_channel=False):
    '''用**平均**误差 / 典型幅值(p90)。两个选择都是刻意的：
       · 用 mean 而非 max —— max 永远由离群值所在的那一格决定，看不出「其余权重被挤坏了多少」
       · 用 p90 而非 max 归一化 —— 用 max 归一化会被离群值自己掩盖'''
    out = {}
    for name, w in weights.items():
        if w.ndim < 2: continue
        w_hat = dequantize(*quantize_int8(w, per_channel))
        typical = float(np.percentile(np.abs(w), 90)) + 1e-12
        out[name] = float(np.abs(w_hat - w).mean() / typical)
    return out

heavy_tail = rng.normal(size=(8, 8)) * 0.3
heavy_tail[0, 0] = 50.0                       # 一个极端离群值 -> 量化范围被它撑爆
weights = {'W1': m.initializers['W1'], 'W2': m.initializers['W2'], 'W_bad': heavy_tail}
errs = per_layer_quant_error(weights, per_channel=False)
med = float(np.median(list(errs.values())))
print(f"{'层':<10s} {'相对量化误差 (per-tensor)':>24s}")
for k, v in errs.items():
    flag = '  ← ⚠️ 异常层' if v > 3 * med else ''
    print(f'{k:<10s} {v:>23.2%}{flag}')
assert errs['W_bad'] > 5 * errs['W1'], '含离群值的层量化误差应显著更大'
print('\n⚠️  一个离群值（50.0）把整层的 scale 撑大了两个数量级 ——')
print('    于是**其余所有权重都被挤到极少数几个量化格子里**。')

errs_pc = per_layer_quant_error(weights, per_channel=True)
print(f'\nW_bad 的误差: per-tensor {errs["W_bad"]:.2%} -> per-channel {errs_pc["W_bad"]:.2%}')
assert errs_pc['W_bad'] < errs['W_bad'] / 3, 'per-channel 应把损害限制在含离群值的那一列'
print('✅ per-channel 把损害**限制在含离群值的那一列**，其余列不受影响。')
print('✅ 所以要**逐层看误差分布**而不是只看端到端 ——')
print('   解法：per-channel 量化、离群值裁剪、或把该层留在 fp16。')

## ✏️ 练习 1：导出前的 opset 与覆盖检查

实现 `export_precheck(model, target_runtime, target_opset)`：返回
`{'opset_ok':…, 'missing_ops': [...], 'partitions':…, 'ok':…}`。
`opset_ok` 要求 **model 里每个算子的 since <= target_opset**，且 `model.opset <= target_opset`。

In [ ]:
def export_precheck(model, target_runtime, target_opset):
    # TODO: ① 检查每个算子的 since <= target_opset 且 model.opset <= target_opset
    #       ② 用 RUNTIME_SUPPORT 求不支持的算子与分区数
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
r = export_precheck(m, 'onnxruntime-cpu', 20)
assert r['opset_ok'] and r['missing_ops'] == [] and r['ok'], r
r2 = export_precheck(m, 'tensorrt', 20)
assert 'Gelu' in r2['missing_ops'] and not r2['ok'] and r2['partitions'] > 1
r3 = export_precheck(m, 'onnxruntime-cpu', 15)
assert not r3['opset_ok'], 'Gelu(since 20) 与 LayerNorm(since 17) 都超过 opset 15'
r4 = export_precheck(decompose_gelu(m), 'tensorrt', 17)
assert r4['ok'], '分解掉 Gelu 后 TensorRT 应可全图支持'
for rt, ops_ in [('onnxruntime-cpu', 20), ('tensorrt', 20), ('tflite', 20)]:
    rr = export_precheck(m, rt, ops_)
    print(f'{rt:<18s} opset_ok={rr["opset_ok"]} missing={rr["missing_ops"]} '
          f'partitions={rr["partitions"]} -> {"✅" if rr["ok"] else "❌"}')
print('✅ 练习 1 通过：**部署前两分钟的检查**，能避免「转完才发现跑不动」')

## ✏️ 练习 2：动态轴规划

实现 `plan_dynamic_axes(input_specs, varying_dims)`：`input_specs` 是
`{名字: (维度名列表)}`，`varying_dims` 是会变化的维度名集合（如 `{'batch','seq'}`）。
返回 `torch.onnx.export` 风格的 `dynamic_axes` 字典 `{名字: {轴下标: 轴名}}`。

In [ ]:
def plan_dynamic_axes(input_specs, varying_dims):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
specs = {'input_ids': ['batch', 'seq'], 'attention_mask': ['batch', 'seq'],
         'pixel_values': ['batch', 'channel', 'height', 'width']}
axes = plan_dynamic_axes(specs, {'batch', 'seq', 'height', 'width'})
assert axes['input_ids'] == {0: 'batch', 1: 'seq'}
assert axes['pixel_values'] == {0: 'batch', 2: 'height', 3: 'width'}, axes['pixel_values']
assert 1 not in axes['pixel_values'], 'channel 不在可变集合里，不应声明为动态'
assert plan_dynamic_axes(specs, set()) == {k: {} for k in specs}, '没有可变维时全为空'
print(json.dumps(axes, ensure_ascii=False, indent=2))
print('✅ 练习 2 通过：**先想清「什么会变」，再导出** —— 事后补是补不上的')

## ✏️ 练习 3：量化容差

实现 `quant_tolerance(bits, per_channel)`：返回建议的 `(rtol, atol)`。
经验规则：`rtol ≈ 2 / (2^(bits-1) - 1)`，per-channel 可以再除以 2；
`atol = rtol`。`bits=16` 时用浮点容差 `1e-3`。

In [ ]:
def quant_tolerance(bits, per_channel=False):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r8, a8 = quant_tolerance(8, per_channel=False)
r8c, _ = quant_tolerance(8, per_channel=True)
r4, _ = quant_tolerance(4)
r16, _ = quant_tolerance(16)
assert abs(r8 - 2/127) < 1e-9, f'int8 per-tensor 应约 {2/127:.4f}，得到 {r8}'
assert r8c < r8, 'per-channel 容差更紧'
assert r4 > r8, '位数越少容差越宽'
assert r16 <= 1e-3
print(f"{'配置':<24s} {'rtol':>10s}")
for bits, pc in [(16, False), (8, False), (8, True), (4, False)]:
    r_, _ = quant_tolerance(bits, pc)
    print(f'{f"int{bits} {"per-channel" if pc else "per-tensor"}":<24s} {r_:>10.5f}')
print('\n✅ 练习 3 通过：**用 fp32 的容差去对拍 int8 结果必然「失败」——但那不是 bug**')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def export_precheck(model, target_runtime, target_opset):
    too_new = [n.op_type for n in model.nodes if OPS[n.op_type]['since'] > target_opset]
    opset_ok = (not too_new) and model.opset <= target_opset
    cov = coverage_report(model, target_runtime)
    return {'opset_ok': opset_ok, 'too_new_ops': sorted(set(too_new)),
            'missing_ops': cov['unsupported'], 'partitions': cov['partitions'],
            'ok': opset_ok and cov['fully_supported']}

In [ ]:
# 练习 2 参考答案
def plan_dynamic_axes(input_specs, varying_dims):
    return {name: {i: d for i, d in enumerate(dims) if d in varying_dims}
            for name, dims in input_specs.items()}

In [ ]:
# 练习 3 参考答案
def quant_tolerance(bits, per_channel=False):
    if bits >= 16:
        return 1e-3, 1e-3
    r = 2.0 / (2 ** (bits - 1) - 1)
    if per_channel:
        r /= 2.0
    return r, r

---
## 🧪 真实 API 对照胶囊：一份可直接用的导出与验证脚本

In [ ]:
RECIPE = r'''
import torch, numpy as np, onnx, onnxruntime as ort

CKPT, ONNX_PATH, OPSET = "my-model", "model.onnx", 17     # ← 先查目标运行时的 opset 上限
model.eval()                                               # ① 关随机性（模块 02 的断言）
with torch.inference_mode():
    a, b = model(example), model(example)
    assert torch.equal(a, b), "还有未关闭的随机性（dropout/BN/自定义随机）"

# ② 导出：**样例输入用非特殊形状**（别用 batch=1）
example = (torch.randint(0, 1000, (2, 17)), torch.ones(2, 17, dtype=torch.long))
torch.onnx.export(
    model, example, ONNX_PATH,
    opset_version=OPSET,
    input_names=["input_ids", "attention_mask"], output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch", 1: "seq"},          # ③ 凡是会变的都声明
                  "attention_mask": {0: "batch", 1: "seq"},
                  "logits": {0: "batch"}},
    do_constant_folding=True,
)

# ④ 结构检查 + 算子清单（部署前两分钟）
m = onnx.load(ONNX_PATH); onnx.checker.check_model(m)
ops = sorted({n.op_type for n in m.graph.node})
print("opset:", m.opset_import[0].version, "| ops:", ops)
# 与目标运行时的支持列表求交集，提前发现「会静默回退」的算子

# ⑤ **多形状**数值对拍（含奇数、极小、非对齐、大 batch）
sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
for bsz, seq in [(1, 128), (2, 17), (8, 512), (3, 333), (1, 1)]:
    ids = torch.randint(0, 1000, (bsz, seq)); mask = torch.ones(bsz, seq, dtype=torch.long)
    with torch.inference_mode():
        ref = model(ids, mask).numpy()
    got = sess.run(None, {"input_ids": ids.numpy(), "attention_mask": mask.numpy()})[0]
    np.testing.assert_allclose(ref, got, rtol=1e-3, atol=1e-4,
                               err_msg=f"mismatch at {bsz}x{seq}")

# ⑥ 检查静默回退（TensorRT / CUDA EP）
sess_gpu = ort.InferenceSession(ONNX_PATH,
                                providers=["TensorrtExecutionProvider", "CPUExecutionProvider"])
print("实际使用的 providers:", sess_gpu.get_providers())
# TensorRT 日志里搜 "subgraph" —— 子图数 > 1 说明发生了回退

# ⑦ 写交付契约（模块 00 的 Contract）
CONTRACT = dict(opset=OPSET, dtype="float32", max_batch=64, max_seq=512,
                tolerance=dict(rtol=1e-3, atol=1e-4),
                notes=["seq 无需对齐（已在 3x333 上验证）",
                       "batch=1 走不同 kernel，延迟不成比例"])
'''
print(RECIPE)
for c in ['torch.equal', 'dynamic_axes', 'opset_version', 'check_model',
          'get_providers', 'CONTRACT', '(2, 17)']:
    assert c in RECIPE, c
print('✅ 配方覆盖：关随机性 / 非特殊样例 / 动态轴 / 结构检查 / 多形状对拍 / 回退检查 / 交付契约')

### 小结
- **导出 = 把 Python 程序编译成受限 IR 上的静态图**。模块 01 的追踪语义全部适用，但在导出里
  它们是**正确性问题**（图交出去后再也不会重新追踪），不只是性能问题。
- **ONNX 的四要素**：静态拓扑、opset 版本、符号形状（动态轴）、内联权重。
- **不设动态轴 = 形状被写死**，「用 batch=1 导出、线上来 batch=8」是最常见的导出事故。
  **别用 batch=1 做样例输入**（它可能走不同代码路径）。
- **算子不支持的四条出路**：换算子 → 提 opset → 分解 → 自定义算子。**出路①成功率最高却常被跳过。**
- **形状依赖数值的算子**（NonZero/Unique/masked_select）在图 IR 里天然困难 →
  **通用解法：把动态部分挪到模型之外。**
- **静默回退**：TensorRT/CoreML/TFLite 遇到不支持的算子不报错，切子图交给别的后端 →
  「转了但没变快」。**看分区日志，不只看端到端延迟。**
- **多形状对拍是必需的**，测试形状要刻意含 batch=1、seq=1、奇数、非对齐、极大值。
- **量化的容差要按量化误差设**（int8 约 2/127），并**逐层找异常层**——一个离群值就能撑爆整层范围。

下一站：**模块 04 · 真机 GPU 工作流** —— 在真实硬件上诊断，而不是猜。